# Week 4 - NLP and Deep Learning

---

# Lecture 7. RNN

In assignments for this lecture we are going to implement an RNN POS tagger in Pytorch.

You can use the following function for data reading:

In [161]:
def read_conll_file(path):
    """
    read in conll file
    
    :param path: path to read from
    :returns: list with sequences of words and labels for each sentence
    """
    data = []
    current_words = []
    current_tags = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split('\t')

            current_words.append(tok[0])
            current_tags.append(tok[1])
        else:
            if current_words:  # skip empty lines
                data.append((current_words, current_tags))
            current_words = []
            current_tags = []

    # check for last one
    if current_tags != []:
        data.append((current_words, current_tags))
    return data

train_data = read_conll_file('pos-data/en_ewt-train.conll')
dev_data = read_conll_file('pos-data/en_ewt-dev.conll')

print(len(train_data))
print(len(dev_data))
print(max([len(x[0]) for x in train_data ]))

12543
2000
159


## 1. Prepare data for use in PyTorch

* a) Convert the data to a format that can be used in a Pytorch module. This means we require:

  * training data: matrix of number of instances (12543) by the maximum sentence length (159), filled with word indices
  * training labels: matrix of the same size in the first dimension, but filled with label indexes instead ( total of 17)
  * the same two sets for the dev data (note that no word indices can be added anymore)
  
A special `<PAD>` token can be used for padding, for sentences shorter as 159 words. For the unknown words in the test set, you can use the `<PAD>` token as well.

**HINT** It will be beneficial in the long run to make a function to convert your data to the right format, as you would have to do it for the train, dev and test sets, and for any other dataset you want to evaluate on.

In [162]:
import torch


class Vocab():
    def __init__(self, pad_unk='<PAD>'):
        """
        A convenience class that can help store a vocabulary
        and retrieve indices for inputs.
        """
        self.pad_unk = pad_unk
        self.word2idx = {self.pad_unk: 0}
        self.idx2word = [self.pad_unk]

    def getIdx(self, word, add=False):
        if word not in self.word2idx:
            if add:
                self.word2idx[word] = len(self.idx2word)
                self.idx2word.append(word)
            else:
                return self.word2idx[self.pad_unk]
        return self.word2idx[word]

    def getWord(self, idx):
        return self.idx2word(idx)

max_len= max([len(x[0]) for x in train_data ])

# Your implementation goes here:
PAD_TOKEN = "<PAD>"
MAX_LEN = 159

word2idx = {PAD_TOKEN: 0}
label2idx = {PAD_TOKEN: 0}

# build vocabulary
for words, labels in train_data:
    for w in words:
        if w not in word2idx:
            word2idx[w] = len(word2idx)

    for l in labels:
        if l not in label2idx:
            label2idx[l] = len(label2idx)

import numpy as np

def encode_dataset(data, word2idx, label2idx, max_len=159):
    
    X = np.zeros((len(data), max_len), dtype=int)
    Y = np.zeros((len(data), max_len), dtype=int)

    for i, (words, labels) in enumerate(data):
        
        for j in range(min(len(words), max_len)):
            
            word = words[j]
            label = labels[j]

            # unknown words → PAD index
            X[i, j] = word2idx.get(word, word2idx["<PAD>"])
            Y[i, j] = label2idx[label]

    return X, Y

X_train, Y_train = encode_dataset(train_data, word2idx, label2idx, MAX_LEN)
X_dev, Y_dev = encode_dataset(dev_data, word2idx, label2idx, MAX_LEN)

import torch

X_train = torch.LongTensor(X_train)
Y_train = torch.LongTensor(Y_train)

X_dev = torch.LongTensor(X_dev)
Y_dev = torch.LongTensor(Y_dev)

X_train.shape

torch.Size([12543, 159])

* b) Until now, we have used a batch-size of 1 in our implemented models, meaning that the models weights are updated after each sentence. This is not very computationally efficient. Larger batch-sizes increase the training speed, and can also lead to better performance (more stable training). You can easily convert existing training data to batches, by splitting it up in chunks of `batch_size` sentences, like this (*Make sure you understand this code!*):

In [163]:
import torch
# 200 instances, 100 features/weights
tmp_feats = torch.zeros((200, 100))

batch_size = 32
num_batches = int(len(tmp_feats)/batch_size)

print(num_batches)

print(tmp_feats.shape)

tmp_feats_batches = tmp_feats[:batch_size*num_batches].view(num_batches,batch_size, 100)

# 6 batches with 32 instances with 100 features
print(tmp_feats_batches.shape)

print()
for feats_batch in tmp_feats_batches:
    print(feats_batch.shape)
    # Here you can call forward/calculate the loss etc.

6
torch.Size([200, 100])
torch.Size([6, 32, 100])

torch.Size([32, 100])
torch.Size([32, 100])
torch.Size([32, 100])
torch.Size([32, 100])
torch.Size([32, 100])
torch.Size([32, 100])


Note that this throws away a tiny part of the data (the last `len(tmp_feats)%batch_size`=6 samples), an alternative would be to pad, and ignore the padded part of the last batch for the loss. For the following assignments you can leave the remaining samples out (note that the dev set is dividable by 16 in this case). Furthermore, note that PyTorch supports a more advanced method for batching: [data loaders](https://pytorch.org/docs/stable/data.html), which we will not cover in this course (but you can use them for the final project).

Convert your training data and labels to batches of batch size 16

In [164]:
# Your implementation goes here:
from torch.utils.data import DataLoader, TensorDataset

batch_size = 16
train_dataset = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

dev_dataset = TensorDataset(X_dev,Y_dev)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=True)

## 2 Train an RNN

* a) Implement a POS tagger model in Pytorch using a [`torch.nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) layer for word representations and a [`torch.nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) layer. You can use a [`torch.nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer for prediction of the label. Train this tagger on the language identification data, and evaluate its performance. Note that during each training step, you now get the predictions and loss on a whole batch directly. Use the following hyperparameters: 5 epochs over the full training data, word embeddings dimension: 100, rnn dimension of 50, learning rate of 0.01 in an [Adam optimizer](https://pytorch.org/docs/stable/optim.html) and a [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

Hints:
* **Set batch_first to true!**, as explained on the [`torch.nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) page. By default the RNN expects the input to be in the shape: `(seq_len, batch, rnn_dim)`, when it is set to true it should be: `(batch, seq_len, rnn_dim)`, which is usually easier to work with.
* Training an RNN is generally much slower compared to the machine learning models we implemented before on this data, so we suggest to start with only a sub-part of the data, like 100 or 1,000 sentences. It is also ok to use only 1,000 sentences for your final model (or use the HPC to train the full model).
* To calculate the cross entropy loss, we need the predictions to be in the first dimension. We can convert the predictions values from our model (32\*159\*18 for 1 batch) to a flatter representation (5088\*18) by using: `.view(BATCH_SIZE * max_len, -1)`. Of course, we also have to adapt the labels from 32\*159 to 5088\*1.

For more information on how to implement a Pytorch module, we refer to the code used to obtain the weights in the assignment of week 3 (`week4/train_ff.py`), and the following tutorial series: https://pytorch.org/tutorials/beginner/nlp/index.html (especially the 2nd and 4th tutorials are relevant). You can use the code below as a starting point

In [165]:
from torch import nn
import torch
torch.manual_seed(0)
DIM_EMBEDDING = 100
RNN_HIDDEN = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.01
EPOCHS = 5

class TaggerModel(torch.nn.Module):
    def __init__(self, nwords, ntags):
        super().__init__()

        self.embedding = nn.Embedding(nwords, DIM_EMBEDDING, padding_idx=0) # Embeding
        self.rnn = nn.RNN(input_size=DIM_EMBEDDING, hidden_size=RNN_HIDDEN, batch_first=True, bidirectional=False) # RNN
        self.fc = nn.Linear(RNN_HIDDEN, ntags) # Linear

    def forward(self, inputData):
        embeds = self.embedding(inputData)              # -> (batch_size, seq_len, DIM_EMBEDDING)
        rnn_out, _ = self.rnn(embeds)                  # -> (batch_size, seq_len, RNN_HIDDEN)
        tag_scores = self.fc(rnn_out)                  # -> (batch_size, seq_len, ntags)
        return tag_scores

model = TaggerModel(len(word2idx), len(label2idx))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_function = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')

for epoch in range(EPOCHS):
    model.train()
    total_loss=0
    # loop over batches
    for X_batch,Y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch) #predicted_values = model.forward()

        outputs = outputs.view(-1, outputs.shape[-1])  # (batch_size*seq_len, ntags)
        Y_batch = Y_batch.view(-1)                      # (batch_size*seq_len)
        # calculate loss (and print)
        loss = loss_function(outputs, Y_batch)
        loss.backward()
        # update
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.2f}")

Epoch 1/5, Loss: 115332.93
Epoch 2/5, Loss: 51050.30
Epoch 3/5, Loss: 35914.27
Epoch 4/5, Loss: 30820.63
Epoch 5/5, Loss: 28576.78


* b) Now evaluate the tagger on the dev data (`pos-data/en_ewt-dev.conll`) with accuracy (make sure to not count the padding tokens).

In [166]:
# Evaluation
model.eval()
pad_idx=0
correct=0
total=0
with torch.no_grad():
    for X_batch, Y_batch in dev_loader:
        outputs = model(X_batch)
        predicted_tags = torch.argmax(outputs, dim=-1)
        
        mask = Y_batch != pad_idx
        correct+=(predicted_tags == Y_batch).masked_select(mask).sum().item()
        total += mask.sum().item()

accuracy = correct / total if total > 0 else 0
print(accuracy)

0.8815595782773026


## 3. Implement a Bi-RNN in Pytorch
In this assignment we are going to implement a bidirectional RNN classifier in Pytorch including dropout layers, and train it for the task of topic classification.

You can use the following function for data reading:

In [167]:
def load_topics(path):
    text = []
    labels = []
    for lineIdx, line in enumerate(open(path)):
        tok = line.strip().split('\t')
        labels.append(tok[0])
        text.append(tok[1].split(' '))
    return text, labels

topic_train_text, topic_train_labels = load_topics('topic-data/train.txt')
topic_dev_text, topic_dev_labels = load_topics('topic-data/dev.txt')

* a) Convert the data to a format that can be used in a Pytorch module. In this assignment, we can cap the size of an utterance, as each utterance only needs 1 label. Use a maximum length of 64 words, for longer sentences, only keep the first 64 words. A special `<PAD>` token can be used for padding for sentences shorter as 64 words. For the unknown words in the test set, you can use the `<PAD>` token as well.

**hint**: the shape of the training data should be 13,000 by 64

In [172]:
PAD_TOKEN = "<PAD>"
MAX_LEN = 64
PAD_IDX = 0

word2idx = {PAD_TOKEN: 0}
for sentence in topic_train_text:
    for word in sentence:
        if word not in word2idx:
            word2idx[word] = len(word2idx)

label2idx = {}  # no PAD for labels
for label in topic_train_labels:
    if label not in label2idx:
        label2idx[label] = len(label2idx)

def encode_words(sentences, word2idx, max_len=64):
    X = []
    for seq in sentences:
        idxs = [word2idx.get(word, PAD_IDX) for word in seq]   # unknown → PAD
        # pad to max_len
        idxs += [PAD_IDX] * (max_len - len(idxs))
        X.append(idxs[:max_len])
    return torch.tensor(X, dtype=torch.long)

X_train = encode_words(topic_train_text, word2idx, MAX_LEN)
X_dev   = encode_words(topic_dev_text,   word2idx, MAX_LEN)

Y_train = torch.tensor([label2idx[l] for l in topic_train_labels], dtype=torch.long)
Y_dev   = torch.tensor([label2idx[l] for l in topic_dev_labels],   dtype=torch.long)

* b) Convert your input into batches of size 32, similar as you did in assignment 1b

In [173]:
batch_size = 32
train_dataset = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

dev_dataset = TensorDataset(X_dev,Y_dev)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=True)

* c) Implement a classification model in Pytorch using a [`torch.nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) layer and a [`torch.nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) layer. Train this classification model on the language identification data, and evaluate its performance. Note that during each training step, you now get the predictions and loss on a whole batch directly. Use the following hyperparameters: 5 epochs over the full training data, word embeddings dimension: 100, rnn dimension of 50, learning rate of 0.01 in an [Adam optimizer](https://pytorch.org/docs/stable/optim.html) and a [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

Hints:
* see also the hints for assignment2a
* Set bidirectional=True for the RNN layer (so that we are training a RNN), note that the input dimensions of the next layer should then be rnn_dim*2. 
* We use words as inputs, and need only one label per sentence, so you should use the output of the last item from the forward layer, and the output of the first item for the backward layer.

In [ ]:
import torch
import torch.nn as nn

class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_dim, num_classes, padding_idx=0, dropout=0.2):
        super(BiRNNClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=rnn_dim,
            batch_first=True,
            bidirectional=True
        )
        # Bidirectional means we concatenate forward and backward hidden states
        self.classifier = nn.Linear(rnn_dim * 2, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))         # (batch, seq_len, embed_dim)
        output, hidden = self.rnn(embedded)                # hidden: (2, batch, rnn_dim)
        
        # Concatenate the final forward and backward hidden states
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)  # (batch, rnn_dim * 2)
        hidden = self.dropout(hidden)
        
        return self.classifier(hidden)                     # (batch, num_classes)


def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch_texts, batch_labels in dataloader:
        batch_texts = batch_texts.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_texts)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        total_correct += (preds == batch_labels).sum().item()
        total_loss += loss.item() * batch_labels.size(0)
        total_samples += batch_labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0

    with torch.no_grad():
        for batch_texts, batch_labels in dataloader:
            batch_texts = batch_texts.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_texts)
            loss = criterion(logits, batch_labels)

            preds = logits.argmax(dim=1)
            total_correct += (preds == batch_labels).sum().item()
            total_loss += loss.item() * batch_labels.size(0)
            total_samples += batch_labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, dev_loader, epochs=5, lr=0.01, device='cpu'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        dev_loss, dev_acc = evaluate(model, dev_loader, criterion, device)

        print(f"Epoch {epoch}/{epochs}")
        print(f"  Train → Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Dev   → Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f}")

    return model

In [177]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BiRNNClassifier(
    vocab_size=len(word2idx),      # your vocab size
    embed_dim=100,              # word embedding dimension
    rnn_dim=50,                 # RNN hidden dimension
    num_classes=len(label2idx), # number of topic classes
    padding_idx=0               # index used for padding tokens
)

trained_model = train_model(
    model, train_loader, dev_loader,
    epochs=5, lr=0.01, device=device
)

Epoch 1/5
  Train → Loss: 0.6713 | Acc: 0.7151
  Dev   → Loss: 0.4691 | Acc: 0.8270
Epoch 2/5
  Train → Loss: 0.3648 | Acc: 0.8746
  Dev   → Loss: 0.4193 | Acc: 0.8400
Epoch 3/5
  Train → Loss: 0.2654 | Acc: 0.9138
  Dev   → Loss: 0.4205 | Acc: 0.8850
Epoch 4/5
  Train → Loss: 0.2751 | Acc: 0.9065
  Dev   → Loss: 0.3819 | Acc: 0.8800
Epoch 5/5
  Train → Loss: 0.2049 | Acc: 0.9343
  Dev   → Loss: 0.2493 | Acc: 0.9200



* d) Add a `torch.nn.Dropout` layer with a masking probability of 0.2 between the word embeddings and the RNN layer and
  another dropout layer with a masking probability of 0.3 between the rnn layer and the output layer. Evaluate the
  performance again, is the performance higher?, why would this be the case?